# Full Pair Comparison

Runs the NashQ, SRQ, and DeepSRQ pair-comparison loop.

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
if notebook_dir.name != "bimatrix_game":
    candidate = notebook_dir / "discrete_action_space" / "bimatrix_game"
    if candidate.exists():
        notebook_dir = candidate.resolve()

for path in (notebook_dir, notebook_dir.parent):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from experiment_harness import (
    BASE_SEED,
    configure_path_runtime,
    run_all_pairings,
    run_deep_srq_epsilon_sweep,
    save_training_stats,
)

discrete_action_space_dir = notebook_dir.parent
pathwrap_path = Path(configure_path_runtime(discrete_action_space_dir)).resolve()
if not pathwrap_path.exists():
    raise FileNotFoundError(f"PATH solver library not found at {pathwrap_path}")
pathwrap_path = str(pathwrap_path)

In [2]:
scenarios = ("scenario1", "scenario2", "scenario3")
pairings = (
    ("NashQ", "NashQ"),
    ("SRQ", "SRQ"),
    ("DeepSRQ", "DeepSRQ"),
    ("NashQ", "SRQ"),
    ("NashQ", "DeepSRQ"),
    ("SRQ", "DeepSRQ"),
)
epsilon_starts = (0.25, 0.5, 1.0)
epsilon_schedules = ("linear", "constant")

solver_name = "path_c_pool"
network_type = "joint_output"  # joint Q network
n_episodes = 3000
batch_size = 16
target_update = 100
use_gpu = True
write_plots = True

deep_srq_hyperparameter_overrides = {
    "sre_num_repeats": 5,
    "sre_include_pure_starts": True,
    "network_type": network_type,
    "sre_solver_workers": 8,
    "sre_solver_start_method": None,
}

sweep_output_root = Path("deep_srq_epsilon_sweep")
pair_comparison_output_root = Path("scenario_runs")

deep_srq_sweep_kwargs = {
    "n_episodes": n_episodes,
    "base_seed": BASE_SEED,
    "scenarios": scenarios,
    "epsilon_starts": epsilon_starts,
    "epsilon_schedules": epsilon_schedules,
    "solver_names": (solver_name,),
    "pathwrap_path": pathwrap_path,
    "output_root": sweep_output_root,
    "use_gpu": use_gpu,
    "write_plots": write_plots,
    "hyperparameter_overrides": deep_srq_hyperparameter_overrides,
}

pair_comparison_kwargs = {
    "scenarios": scenarios,
    "pairings": pairings,
    "n_episodes": n_episodes,
    "pathwrap_path": pathwrap_path,
    "solver_name": solver_name,
    "output_root": pair_comparison_output_root,
    "batch_size": batch_size,
    "target_update": target_update,
    "use_gpu": use_gpu,
    "write_plots": write_plots,
    "hyperparameter_overrides": deep_srq_hyperparameter_overrides,
}

## Full Deep SRQ Epsilon Sweep

In [ ]:
deep_srq_sweep_results = run_deep_srq_epsilon_sweep(**deep_srq_sweep_kwargs)

save_training_stats(
    sweep_output_root / "full_deep_srq_epsilon_sweep_manifest.txt",
    deep_srq_sweep_results,
)

Process ForkPoolWorker-1:
Traceback (most recent call last):
Process ForkPoolWorker-2:
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Traceback (most recent call last):
  File "/home/wowthecoder/SRE-DQN/discrete_action_space/sre_solvers/path_c.py", line 91, in __init__
    self.path_solver = PathSolverWrapper(pathwrap_path)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/pool.py", line 109, in worker
    initializer(*initargs)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/wowthecoder/SRE-DQN/discrete_action_space/sre_solvers/path_c.py", line 165, in _path_pool_initializer
    _POOL_SOLVER = PathCBimatrixSreSolver(pathwrap_path=pathwrap_path)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## Full Pair Loop

In [ ]:
results = run_all_pairings(**pair_comparison_kwargs)

save_training_stats(
    pair_comparison_output_root / "full_pair_comparison_manifest.txt",
    results,
)